In [ ]:
import os
import re
import warnings
import pandas as pd
import pg8000
from openai import OpenAI
from IPython.display import display, clear_output

try:
    import matplotlib
    try:
        ip = get_ipython()
        if ip is not None:
            ip.run_line_magic("matplotlib", "inline")
    except Exception:
        pass

    if os.environ.get("MPLBACKEND") is None:
        try:
            matplotlib.use("module://matplotlib_inline.backend_inline")
        except Exception:
            pass

    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

warnings.filterwarnings(
    'ignore',
    message='pandas only supports SQLAlchemy connectable',
    category=UserWarning,
)


In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

conn = pg8000.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME", "postgres"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
PALABRAS_MAPEO = {
    "accion": "Action",
    "aventura": "Adventure",
    "deportes": "Sports",
    "rpg": "RPG",
    "pc": "PC",
    "playstation": "PlayStation",
    "xbox": "Xbox"
}


In [ ]:
COLUMNAS_VALIDAS = {
    "name", "slug", "released", "rating", "ratings_count",
    "reviews_text_count", "metacritic", "added",
    "suggestions_count", "genres", "platforms",
    "developers", "publishers", "id"
}

In [ ]:
def construir_prompt(pregunta):
    for k, v in PALABRAS_MAPEO.items():
        pregunta = pregunta.lower().replace(k, v)

    return f"""
Eres un experto en SQL para PostgreSQL.

Tabla: juego

Columnas disponibles:
name, slug, released, rating, ratings_count, reviews_text_count,
metacritic, added, suggestions_count,
genres, platforms, developers, publishers, id

IMPORTANTE:
Las columnas genres, platforms, developers y publishers contienen texto con múltiples valores.

SIEMPRE usar:
ILIKE '%valor%'

NO usar =
NO inventar columnas

Reglas:
- SOLO SELECT
- SOLO tabla juego
- SIEMPRE LIMIT 100

Ejemplos:

SELECT name, rating
FROM juego
WHERE genres ILIKE '%Action%'
ORDER BY rating DESC
LIMIT 100;

---

Pregunta: {pregunta}

Devuelve SOLO SQL.
"""

In [ ]:
def limpiar_sql(texto):
    if texto is None:
        return ""

    s = texto.strip()

    s = re.sub(r"^```\s*sql\s*\n", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^```\s*\n", "", s)
    s = re.sub(r"\n```$", "", s)

    s = s.strip()

    s = re.sub(
        r"extract\s*\(\s*year\s+from\s+released\s*\)",
        "EXTRACT(YEAR FROM released::date)",
        s,
        flags=re.IGNORECASE,
    )

    s = re.sub(
        r"\b(genres|platforms|developers|publishers)\s+(ilike|like)\b",
        r"\1::text \2",
        s,
        flags=re.IGNORECASE,
    )

    return s


def generar_sql(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return limpiar_sql(response.choices[0].message.content)

In [ ]:
def validar_sql(sql):
    sql_lower = sql.lower().strip()

    if not sql_lower.startswith(("select", "with")):
        return False, "Solo SELECT permitido"

    prohibidas = [
        "insert", "update", "delete", "drop", "alter", "truncate",
        "create", "grant", "revoke", "comment", "vacuum"
    ]
    if any(re.search(rf"\b{kw}\b", sql_lower) for kw in prohibidas):
        return False, "Operacion no permitida"

    if "limit" not in sql_lower:
        return False, "Falta LIMIT"

    if not re.search(r"\bfrom\b", sql_lower) or "juego" not in sql_lower:
        return False, "Solo tabla juego"

    sql_sin_strings = re.sub(r"'([^']|'')*'", "''", sql_lower)
    palabras = re.findall(r"\b\w+\b", sql_sin_strings)

    palabras_permitidas = {
        "select","from","where","and","or","group","by","order",
        "limit","avg","count","min","max","sum",
        "desc","asc","ilike","like","in","between",
        "distinct","as","having","null","is","not","coalesce",
        "case","when","then","else","end",
        "date","cast","extract","year",
        "text","json","jsonb",
        "juego"
    }

    for i, p in enumerate(palabras):
        if i > 0 and palabras[i - 1] == "as":
            continue

        if p not in COLUMNAS_VALIDAS and p not in palabras_permitidas and not p.isdigit():
            return False, f"Palabra no válida: {p}"

    return True, ""

In [ ]:
def aplicar_filtro_anios(sql, min_year, max_year):
    filtro = f"released::date BETWEEN '{min_year}-01-01'::date AND '{max_year}-12-31'::date"

    if re.search(r"\bwhere\b", sql, flags=re.IGNORECASE):
        return re.sub(r"\bwhere\b", f"WHERE {filtro} AND ", sql, count=1, flags=re.IGNORECASE)

    m = re.search(
        r"\bfrom\s+juego\b(?:\s+(?!where\b|order\b|group\b|limit\b|having\b|join\b)\w+)?",
        sql,
        flags=re.IGNORECASE,
    )
    if m:
        insert_at = m.end()
        return sql[:insert_at] + f" WHERE {filtro}" + sql[insert_at:]

    return sql

In [ ]:
def detectar_columnas(df):

    numericas = df.select_dtypes(include=["float64", "int64"]).columns.tolist()

    categoricas = df.select_dtypes(include=["object", "string"]).columns.tolist()
    return numericas, categoricas


def generar_conclusion(df, pregunta):
    p = (pregunta or "").lower()

    if "year" in df.columns:
        metricas = [c for c in df.columns if c != "year" and pd.api.types.is_numeric_dtype(df[c])]
        if metricas:
            y_col = metricas[0]
            tmp = df[["year", y_col]].dropna().copy()
            if len(tmp) >= 2:
                tmp["year"] = tmp["year"].astype(int)
                tmp = tmp.sort_values("year")
                y0 = float(tmp.iloc[0][y_col])
                y1 = float(tmp.iloc[-1][y_col])
                tendencia = "aumenta" if y1 > y0 else "disminuye" if y1 < y0 else "se mantiene"

                idx_max = tmp[y_col].idxmax()
                anio_max = int(tmp.loc[idx_max, "year"])
                val_max = float(tmp.loc[idx_max, y_col])

                return (
                    f"Entre {int(tmp.iloc[0]['year'])} y {int(tmp.iloc[-1]['year'])}, {y_col} {tendencia}. "
                    f"El máximo se da en {anio_max} con {val_max:.0f}."
                )

    if "name" in df.columns:
        metricas_candidatas = [
            c for c in df.columns
            if c != "name" and pd.api.types.is_numeric_dtype(df[c])
        ]
        if metricas_candidatas:
            y_col = metricas_candidatas[0]
            tmp = df[["name", y_col]].dropna().copy()
            if len(tmp) > 0:
                top_row = tmp.sort_values(y_col, ascending=False).iloc[0]
                nombre = str(top_row["name"])
                valor = float(top_row[y_col])
                return f"El juego líder es '{nombre}' con {y_col} = {valor:.2f}."

    def top_frecuente(col):
        s = df[col].fillna("").astype(str)
        parts = (
            s.str.replace("[", "", regex=False)
            .str.replace("]", "", regex=False)
            .str.replace("'", "", regex=False)
            .str.split(",")
        )
        vals = parts.explode().str.strip()
        vals = vals[vals.ne("")]
        if len(vals) == 0:
            return None
        top = vals.value_counts().head(1)
        return top.index[0], int(top.iloc[0])

    if ("categoria" in p or "categoría" in p or "genero" in p or "género" in p) and "genres" in df.columns:
        res = top_frecuente("genres")
        if res:
            genero, cnt = res
            return f"El género más frecuente en el periodo es '{genero}' con {cnt} apariciones."

    if ("plataforma" in p or "platform" in p) and "platforms" in df.columns:
        res = top_frecuente("platforms")
        if res:
            plat, cnt = res
            return f"La plataforma más frecuente en el periodo es '{plat}' con {cnt} apariciones."

    return f"Se han obtenido {len(df)} filas para esta consulta."

In [ ]:
def generar_grafico(df, pregunta):
    if plt is None:
        print('matplotlib no esta instalado en el kernel; no se puede generar grafico.')
        return

    try:
        plt.close('all')
    except Exception:
        pass

    num, cat = detectar_columnas(df)

    if len(df) == 0:
        print("No hay datos")
        return

    p = (pregunta or "").lower()

    def extraer_top_n(texto, default=10):
        m = re.search(r"top\s*(\d+)", texto or "", flags=re.IGNORECASE)
        if m:
            try:
                return max(1, int(m.group(1)))
            except ValueError:
                pass
        return default

    def split_multivalor(series):
        s = series.fillna("").astype(str)
        partes = (
            s.str.replace("[", "", regex=False)
            .str.replace("]", "", regex=False)
            .str.replace("'", "", regex=False)
            .str.split(",")
        )
        valores = partes.explode().str.strip()
        return valores[valores.ne("")]

    if ("categoria" in p or "categoría" in p or "genero" in p or "género" in p) and "genres" in df.columns:
        top_n = extraer_top_n(p, default=10)
        generos = split_multivalor(df["genres"])

        if ("año" in p or "anio" in p) and "released" in df.columns:
            fechas = pd.to_datetime(df["released"], errors="coerce")
            anios = fechas.dt.year

            tmp = pd.DataFrame({"year": anios, "genre": generos.reset_index(drop=True)})
            tmp = tmp.dropna(subset=["year", "genre"])
            tmp["year"] = tmp["year"].astype(int)

            counts = tmp.groupby(["year", "genre"]).size().rename("count").reset_index()
            if len(counts) == 0:
                print("No hay datos de año/género para graficar")
                return

            top_genres = (
                counts.groupby("genre")["count"].sum().sort_values(ascending=False).head(top_n).index
            )
            counts = counts[counts["genre"].isin(top_genres)]

            pivot = counts.pivot(index="year", columns="genre", values="count").fillna(0).sort_index()
            ax = pivot.plot(figsize=(12, 6), marker="o", title=f"Top {top_n} géneros por año")
            ax.set_xlabel("Año")
            ax.set_ylabel("Cantidad")
            plt.tight_layout()
            plt.show()
            try:
                plt.close(ax.figure)
            except Exception:
                pass
            return

        top = generos.value_counts().head(max(1, top_n))
        if len(top) == 0:
            print("No hay categorías/géneros para graficar")
            return

        ax = top.sort_values().plot.barh(figsize=(10, 6), title=f"Top {top_n} géneros")
        ax.set_xlabel("Cantidad")
        ax.set_ylabel("Género")
        plt.tight_layout()
        plt.show()
        try:
            plt.close(ax.figure)
        except Exception:
            pass
        return

    if ("plataforma" in p or "platform" in p) and "platforms" in df.columns:
        top_n = extraer_top_n(p, default=10)
        plats = split_multivalor(df["platforms"])

        if ("año" in p or "anio" in p) and "released" in df.columns:
            fechas = pd.to_datetime(df["released"], errors="coerce")
            anios = fechas.dt.year

            tmp = pd.DataFrame({"year": anios, "platform": plats.reset_index(drop=True)})
            tmp = tmp.dropna(subset=["year", "platform"])
            tmp["year"] = tmp["year"].astype(int)

            counts = tmp.groupby(["year", "platform"]).size().rename("count").reset_index()
            if len(counts) == 0:
                print("No hay datos de año/plataforma para graficar")
                return

            top_plats = (
                counts.groupby("platform")["count"].sum().sort_values(ascending=False).head(top_n).index
            )
            counts = counts[counts["platform"].isin(top_plats)]

            pivot = counts.pivot(index="year", columns="platform", values="count").fillna(0).sort_index()
            ax = pivot.plot(figsize=(12, 6), marker="o", title=f"Top {top_n} plataformas por año")
            ax.set_xlabel("Año")
            ax.set_ylabel("Cantidad")
            plt.tight_layout()
            plt.show()
            try:
                plt.close(ax.figure)
            except Exception:
                pass
            return

        top = plats.value_counts().head(max(1, top_n))
        if len(top) == 0:
            print("No hay plataformas para graficar")
            return

        ax = top.sort_values().plot.barh(figsize=(10, 6), title=f"Top {top_n} plataformas")
        ax.set_xlabel("Cantidad")
        ax.set_ylabel("Plataforma")
        plt.tight_layout()
        plt.show()
        try:
            plt.close(ax.figure)
        except Exception:
            pass
        return

    if ("top" in p or "mejor" in p) and num and cat:
        x_col = cat[0]
        y_col = num[0]

        df_plot = df[[x_col, y_col]].dropna().copy()
        if len(df_plot) == 0:
            print("No hay datos")
            return

        df_plot = df_plot.sort_values(by=y_col, ascending=False).head(15)
        ax = df_plot.sort_values(by=y_col).plot.barh(x=x_col, y=y_col, figsize=(10, 6), legend=False)
        ax.set_xlabel(y_col)
        ax.set_ylabel(x_col)
        plt.tight_layout()
        plt.show()
        try:
            plt.close(ax.figure)
        except Exception:
            pass
        return

    if "evolucion" in p or "evolución" in p:
        if "year" in df.columns:
            metricas = [c for c in df.columns if c != "year" and pd.api.types.is_numeric_dtype(df[c])]
            if metricas:
                y_col = metricas[0]
                df_plot = df[["year", y_col]].dropna().copy()
                df_plot["year"] = df_plot["year"].astype(int)
                df_plot = df_plot.sort_values("year")
                ax = df_plot.plot(x="year", y=y_col, kind="line", marker="o", figsize=(10, 4), legend=False)
                ax.set_xlabel("Año")
                ax.set_ylabel(y_col)
                plt.tight_layout()
                plt.show()
                try:
                    plt.close(ax.figure)
                except Exception:
                    pass
                return

        if num:
            ax = df[num[0]].plot.line(figsize=(10, 4))
            plt.tight_layout()
            plt.show()
            try:
                plt.close(ax.figure)
            except Exception:
                pass
        return

    if "released" in df.columns:
        fechas = pd.to_datetime(df["released"], errors="coerce")
        anios = fechas.dt.year.dropna().astype(int)
        if len(anios) > 0:
            counts = anios.value_counts().sort_index()
            ax = counts.plot(kind="bar", figsize=(10, 4), title="Número de juegos por año")
            ax.set_xlabel("Año")
            ax.set_ylabel("Número de juegos")
            plt.tight_layout()
            plt.show()
            try:
                plt.close(ax.figure)
            except Exception:
                pass
            return

    if num:
        ax = df[num[0]].plot.hist(figsize=(8, 4), bins=20)
        plt.tight_layout()
        plt.show()
        try:
            plt.close(ax.figure)
        except Exception:
            pass


In [ ]:
def ejecutar(pregunta, min_year, max_year, output):
    with output:
        clear_output()

        p = (pregunta or "")
        p_lower = p.lower()

        es_categoria = any(x in p_lower for x in ["categoria", "categoría", "genero", "género"])
        es_plataforma = any(x in p_lower for x in ["plataforma", "platform", "platforms"])

        plataforma_objetivo = None
        for k, v in PALABRAS_MAPEO.items():
            if v in {"PC", "PlayStation", "Xbox"}:
                if re.search(rf"\b{re.escape(k)}\b", p_lower):
                    plataforma_objetivo = v
                    break

        if es_categoria:
            if "año" in p_lower or "anio" in p_lower:
                sql = "SELECT released, genres FROM juego LIMIT 100;"
            else:
                sql = "SELECT genres FROM juego LIMIT 100;"

        elif es_plataforma:
            if plataforma_objetivo:
                sql = (
                    "SELECT name, platforms "
                    "FROM juego "
                    f"WHERE platforms ILIKE '%{plataforma_objetivo}%' "
                    "LIMIT 100;"
                )
            else:
                if "año" in p_lower or "anio" in p_lower:
                    sql = "SELECT released, platforms FROM juego LIMIT 100;"
                else:
                    sql = "SELECT platforms FROM juego LIMIT 100;"

        else:
            prompt = construir_prompt(p)
            sql = generar_sql(prompt)

            valido, error = validar_sql(sql)
            if not valido:
                print("Error:", error)
                print(sql)
                return

        sql = aplicar_filtro_anios(sql, min_year, max_year)

        print("SQL generado:\n", sql)

        try:
            df = pd.read_sql(sql, conn)
        except Exception:
            try:
                conn.rollback()
            except Exception:
                pass
            raise

        display(df.head())

        generar_grafico(df, p)

        conclusion = generar_conclusion(df, p)
        if conclusion:
            print("\nConclusión:", conclusion)


In [ ]:
if widgets is not None:
    pregunta_input = widgets.Text(placeholder='Ej: mejores juego de accion en pc')
    year_slider = widgets.IntRangeSlider(value=[2000, 2020], min=1980, max=2025, step=1, description='Años')
    boton = widgets.Button(description='Generar')
    output = widgets.Output()
else:
    pregunta_input = None
    year_slider = None
    boton = None
    output = None


In [ ]:
def on_click(b):
    if output is None:
        return
    ejecutar(pregunta_input.value, year_slider.value[0], year_slider.value[1], output)


In [ ]:
if boton is not None:
    boton.on_click(on_click)


In [ ]:
if boton is not None:
    if not bool(globals().get("run_api", False)):
        display(pregunta_input, year_slider, boton, output)


In [ ]:
question = globals().get("question", "")
min_year = int(globals().get("min_year", 2000))
max_year = int(globals().get("max_year", 2020))
run_api = bool(globals().get("run_api", False))


def ejecutar_api(pregunta, min_year, max_year):
    p = (pregunta or "")
    p_lower = p.lower()

    es_categoria = any(x in p_lower for x in ["categoria", "categoría", "genero", "género"])
    es_plataforma = any(x in p_lower for x in ["plataforma", "platform", "platforms"])

    plataforma_objetivo = None
    for k, v in PALABRAS_MAPEO.items():
        if v in {"PC", "PlayStation", "Xbox"}:
            if re.search(rf"\b{re.escape(k)}\b", p_lower):
                plataforma_objetivo = v
                break

    if es_categoria:
        if "año" in p_lower or "anio" in p_lower:
            sql = "SELECT released, genres FROM juego LIMIT 100;"
        else:
            sql = "SELECT genres FROM juego LIMIT 100;"

    elif es_plataforma:
        if plataforma_objetivo:
            sql = (
                "SELECT name, platforms "
                "FROM juego "
                f"WHERE platforms ILIKE '%{plataforma_objetivo}%' "
                "LIMIT 100;"
            )
        else:
            if "año" in p_lower or "anio" in p_lower:
                sql = "SELECT released, platforms FROM juego LIMIT 100;"
            else:
                sql = "SELECT platforms FROM juego LIMIT 100;"

    else:
        prompt = construir_prompt(p)
        sql = generar_sql(prompt)

        valido, error = validar_sql(sql)
        if not valido:
            print("Error:", error)
            print(sql)
            return

    sql = aplicar_filtro_anios(sql, min_year, max_year)

    try:
        df = pd.read_sql(sql, conn)
    except Exception:
        try:
            conn.rollback()
        except Exception:
            pass
        raise


    generar_grafico(df, p)

    conclusion = generar_conclusion(df, p)
    if conclusion:
        print("\nConclusión:", conclusion)

if run_api:
    if not str(question).strip():
        print("Falta 'question' para visualización")
    else:
        ejecutar_api(question, min_year, max_year)